

**Install Libraries**


In [ ]:

!pip install datasets

In [ ]:
from datasets import load_dataset

**Load Dataset**

In [ ]:
dataset = load_dataset(
    "CShorten/ML-ArXiv-Papers",
    split="train"
)

In [ ]:
print(dataset)

In [ ]:
dataset[0]

**dataset into a Pandas DataFrame.**

In [ ]:
import pandas as pd

df = pd.DataFrame(dataset)

df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df = df[['title', 'abstract']]

In [ ]:
df.head()

In [ ]:
df = df.head(15000)

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df["paper_text"] = df["title"] + " " + df["abstract"]

In [ ]:
df[["paper_text"]].head()

In [ ]:
type(df[["paper_text"]])

In [ ]:
df["paper_text"].head()

In [ ]:
type(df["paper_text"])

In [ ]:
print(df["paper_text"].iloc[0])

In [ ]:
df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex=False)
df["paper_text"] = df["paper_text"].str.strip()

In [ ]:
print(df["paper_text"].iloc[0])

**Sentence Transformers**

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
print(type(model))

In [ ]:
sample_text = df["paper_text"].iloc[0]

embedding = model.encode(sample_text)

print(type(embedding))
print(embedding.shape)

In [ ]:
embedding[:10]

In [ ]:
sample_embeddings = model.encode(
    df["paper_text"].head(5).tolist()
)

print(type(sample_embeddings))
print(sample_embeddings.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(
    sample_embeddings[0].reshape(1, -1),
    sample_embeddings[1].reshape(1, -1)
)

print(similarity)

In [ ]:
similarity = cosine_similarity(
    sample_embeddings[0].reshape(1,-1),
    sample_embeddings[0].reshape(1,-1)
)

print(similarity)

In [ ]:
for i in range(1,5):

    sim = cosine_similarity(
        sample_embeddings[0].reshape(1,-1),
        sample_embeddings[i].reshape(1,-1)
    )

    print(f"Paper 1 vs Paper {i+1}: {sim[0][0]:.4f}")

**Generate Full Embeddings**

In [ ]:
import os
import numpy as np

if os.path.exists("paper_embeddings.npy"):

    print("Loading saved embeddings...")

    embeddings = np.load("paper_embeddings.npy")

else:

    print("Generating embeddings...")

    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )

    np.save("paper_embeddings.npy", embeddings)

    print("Embeddings saved successfully!")#embeddings = model.encode(
    #df["paper_text"].tolist(),
    #batch_size=32,
    #show_progress_bar=True
#)

In [ ]:
print(type(embeddings))
print(embeddings.shape)


In [ ]:
embeddings.nbytes / (1024*1024)

In [ ]:
embeddings.dtype


In [ ]:
!pip install faiss-cpu

In [ ]:
import os
import faiss

if os.path.exists("paper_faiss.index"):

    print("Loading existing FAISS index...")

    index = faiss.read_index("paper_faiss.index")

else:

    print("Creating new FAISS index...")

    faiss.normalize_L2(embeddings)

    index = faiss.IndexFlatIP(384)

    index.add(embeddings)

    faiss.write_index(index, "paper_faiss.index")

    print("FAISS index saved successfully!")#faiss.normalize_L2(embeddings)

#index = faiss.IndexFlatIP(384)

#index.add(embeddings)

In [ ]:
import numpy as np

np.linalg.norm(embeddings[0])

In [ ]:
print(index.ntotal)

In [ ]:
query = "deep learning for medical image analysis"
query_embedding = model.encode([query])
query_embedding.shape


In [ ]:
faiss.normalize_L2(query_embedding)


np.linalg.norm(query_embedding[0])

In [ ]:
D, I = index.search(query_embedding, 5)

print(D)
print(I)

In [ ]:
print(df.iloc[10466]["title"])

In [ ]:
print(df.iloc[10466]["abstract"])

In [ ]:
def search_papers(query, k=5):

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k)

    return D, I

In [ ]:
D, I = search_papers(
    "deep learning for medical image analysis"
)

print(D)
print(I)

In [ ]:
def search_papers(query, k=5):

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):

        print("=" * 100)

        print("Similarity:", round(float(score), 4))

        print("Title:", df.iloc[idx]["title"])

        print()

        print("Abstract:")

        print(df.iloc[idx]["abstract"][:500])

        print()

In [ ]:
search_papers(
    "deep learning for medical image analysis"
)

In [ ]:
!pip install \
transformers==4.46.3 \
huggingface_hub==0.26.2 \
tokenizers==0.20.3 \
sentence-transformers==3.3.1

In [ ]:
import transformers
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Transformers Path:", transformers.__file__)

print("Hub:", huggingface_hub.__version__)
print("Hub Path:", huggingface_hub.__file__)

import transformers.utils as u

print(hasattr(u, "HUGGINGFACE_CO_RESOLVE_ENDPOINT"))

In [ ]:
import transformers.utils as u

print(hasattr(u, "HUGGINGFACE_CO_RESOLVE_ENDPOINT"))

In [ ]:
from transformers import pipeline

In [ ]:
type(pipeline)

In [ ]:
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

In [ ]:
type(summarizer)

In [ ]:
summary = summarizer(
    df.iloc[10466]["abstract"],
    max_length=120,
    min_length=40,
    do_sample=False
)

In [ ]:
print(summary)

In [ ]:
type(summary)

In [ ]:
summary[0]

In [ ]:
type(summary[0])

In [ ]:
summary[0]["summary_text"]

In [ ]:
print(summary[0]["summary_text"])

In [ ]:
for score, idx in zip(D[0], I[0]):

    print("=" * 100)

    print("Similarity:", round(float(score), 4))

    print("Title:", df.iloc[idx]["title"])

    print()

    summary = summarizer(
        df.iloc[idx]["abstract"],
        max_length=120,
        min_length=40,
        do_sample=False
    )

    print("Summary:")

    print(summary[0]["summary_text"])

    print()

In [ ]:
def summarize_text(text):

    summary = summarizer(
        text,
        max_length=120,
        min_length=40,
        do_sample=False
    )

    return summary[0]["summary_text"]

In [ ]:
def search_and_summarize(query, k=5):

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):

        print("=" * 100)

        print("Similarity:", round(float(score), 4))

        print("Title:", df.iloc[idx]["title"])

        print()

        summary = summarizer(
            df.iloc[idx]["abstract"],
            max_length=120,
            min_length=40,
            do_sample=False
        )

        print("Summary:")

        print(summary[0]["summary_text"])

        print()

In [ ]:
search_and_summarize(
    "Deep Learning in Medical Imaging",
    k=5
)

In [ ]:
!pip install keybert==0.8.5

In [ ]:
from keybert import KeyBERT

In [ ]:
kw_model = KeyBERT(model)

In [ ]:
type(kw_model)

In [ ]:
text = df.iloc[10466]["abstract"]

keywords = kw_model.extract_keywords(text)

In [ ]:
print(type(keywords))

In [ ]:
print(type(keywords[0]))

In [ ]:
print(keywords[0])

In [ ]:
keywords = kw_model.extract_keywords(
    text,
    keyphrase_ngram_range=(1,3),
    stop_words="english",
    top_n=5
)

print(keywords)

In [ ]:
def search_and_summarize(query, k=5):

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):

        print("=" * 100)

        print("Similarity:", round(float(score), 4))

        print("Title:", df.iloc[idx]["title"])

        print()

        summary = summarizer(
            df.iloc[idx]["abstract"],
            max_length=120,
            min_length=40,
            do_sample=False
        )

        print("Summary:")

        print(summary[0]["summary_text"])

        print()

        keywords = kw_model.extract_keywords(
            df.iloc[idx]["abstract"],
            keyphrase_ngram_range=(1,3),
            stop_words="english",
            top_n=5
        )

        print("Keywords:")

        for keyword, score in keywords:
            print("-", keyword)

        print()

In [ ]:
search_and_summarize(
    "Deep Learning in Medical Imaging",
    k=5
)

**Hybrid NER Architecture**

In [ ]:
ner = pipeline(
    "ner",
    aggregation_strategy="simple"
)

In [ ]:
text = """
ResNet50 was trained on ImageNet using PyTorch
at Stanford University.
"""
entities = ner(text)

print(entities)

In [ ]:
!pip install langchain langchain-community langchain-core

In [ ]:

from langchain_core.tools import tool
@tool
def search_papers(query: str) -> str:
    """
    Search research papers from the FAISS database
    and return the most relevant results.
    """

In [ ]:
!pip install -U langchain langchain-community langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFacePipeline

In [ ]:
llm = HuggingFacePipeline(
    pipeline=summarizer
)

In [ ]:
llm

In [ ]:
response = llm.invoke(
    "Deep Learning has become one of the most successful techniques for medical image reconstruction using convolutional neural networks."
)

In [ ]:
response

In [ ]:
print(type(response))

In [ ]:
!pip install -U langchain-groq

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
from google.colab import userdata

In [ ]:
api_key = userdata.get("GROQ_API_KEY")

In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=api_key,
    temperature=0
)

In [ ]:
response = llm.invoke("Hello, who are you?")

In [ ]:
response

In [ ]:
response.content

In [ ]:
from langchain_core.tools import tool

In [ ]:


@tool
def search_and_summarize(query: str, k: int = 5) -> str:
    """
    Search research papers from the FAISS database,
    retrieve the top-k most similar papers,
    summarize each paper using BART,
    and return the results.
    """


    query_embedding = model.encode([query])


    faiss.normalize_L2(query_embedding)


    D, I = index.search(query_embedding, k)


    result = ""

    for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1):

        paper = df.iloc[idx]


        summary = summarizer(
            paper["abstract"],
            max_length=120,
            min_length=40,
            do_sample=False
        )[0]["summary_text"]


        result += f"Rank: {rank}\n"
        result += f"Similarity Score: {round(float(score),4)}\n"
        result += f"Title: {paper['title']}\n\n"


        result += paper["abstract"] + "\n\n"


        result += summary + "\n\n"

    return result

In [ ]:
tools = [search_and_summarize]

In [ ]:
from langchain.agents import create_agent

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools
)

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Find the top 3 research papers on Vision Transformer."
            }
        ]
    }
)

In [ ]:
response

In [ ]:
print(type(response))

In [ ]:
print(response.keys())

In [ ]:
len(response["messages"])

In [ ]:
response["messages"][0]

In [ ]:
response["messages"][1]

In [ ]:
response["messages"][2]

In [ ]:
response["messages"][3]

In [ ]:


@tool
def search_and_summarize(query: str, k: int = 5):
    """
    Search research papers from the FAISS database,
    retrieve the top-k most similar papers,
    summarize each paper using BART,
    and return the results as structured data.
    """

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)


    D, I = index.search(query_embedding, k)

    papers = []

    for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1):

        paper = df.iloc[idx]

        summary = summarizer(
            paper["abstract"],
            max_length=120,
            min_length=40,
            do_sample=False
        )[0]["summary_text"]


        paper_data = {
            "rank": rank,
            "similarity": round(float(score), 4),
            "title": paper["title"],
            "abstract": paper["abstract"],
            "summary": summary
        }

        papers.append(paper_data)

    return papers

In [ ]:


@tool
def extract_keywords(text: str, top_n: int = 5) -> str:
    """
    Extract the most important keywords from the given text
    using the KeyBERT model.
    """


    keywords = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 2),
        stop_words="english",
        top_n=top_n
    )

    result = "Top Keywords:\n\n"

    for rank, (keyword, score) in enumerate(keywords, start=1):

        result += (
            f"{rank}. {keyword} "
            f"(Relevance Score: {round(score, 4)})\n"
        )

    return result

In [ ]:
tools = [
    search_and_summarize,
    extract_keywords
]

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools
)

In [ ]:
print(tools)

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract the top 5 keywords from Deep Learning for Medical Image Reconstruction."
            }
        ]
    }
)

In [ ]:
response = extract_keywords.invoke(
    {
        "text": "Deep Learning for Medical Image Reconstruction",
        "top_n": 5
    }
)

print(response)

In [ ]:

user_query = "Extract the top 5 keywords from Deep Learning for Medical Image Reconstruction."

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke(user_query)

print(response)



print(response.tool_calls)




tool_call = response.tool_calls[0]

print(tool_call)

In [ ]:
tool_name = tool_call["name"]

print(tool_name)

In [ ]:
tool_args = tool_call["args"]

print(tool_args)

In [ ]:
if tool_name == "extract_keywords":

    tool_result = extract_keywords.invoke(tool_args)

elif tool_name == "search_and_summarize":

    tool_result = search_and_summarize.invoke(tool_args)

print(tool_result)

In [ ]:
from langchain_core.messages import ToolMessage

In [ ]:
tool_message = ToolMessage(
    content=tool_result,
    tool_call_id=tool_call["id"]
)

In [ ]:
print(tool_message)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

final_response = llm.invoke(
    [
        SystemMessage(
            content="""
You are a helpful AI assistant.

Rules:
1. Always use the tool output.
2. Never ignore tool results.
3. Present the complete tool output.
4. Add a short explanation after the tool output if necessary.
"""
        ),

        HumanMessage(content=user_query),

        response,

        tool_message
    ]
)

print(final_response.content)

In [ ]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage
)

final_response = llm.invoke(
[
SystemMessage(
content="""
You are a research assistant.

IMPORTANT RULES

1. Never ignore tool output.

2. Always display every keyword returned by the tool.

3. Do not summarize the keywords.

4. Print the keywords exactly as received.

5. After printing them, give a short explanation.
"""
),

HumanMessage(content=user_query),

response,

tool_message

]
)

print(final_response.content)

from langchain_core.tools import tool

@tool
def compare_papers(
    paper1: str,
    paper2: str
) -> str:
    """
    Compare two research papers based on their abstracts.
    The tool retrieves the closest matching papers from the FAISS database
    and uses the LLM to compare them.
    """


    embedding1 = model.encode([paper1])

    faiss.normalize_L2(embedding1)

    D1, I1 = index.search(embedding1, 1)

    first_paper = df.iloc[I1[0][0]]


    embedding2 = model.encode([paper2])

    faiss.normalize_L2(embedding2)

    D2, I2 = index.search(embedding2, 1)

    second_paper = df.iloc[I2[0][0]]


    comparison_prompt = f"""
Compare the following two research papers.

Paper 1

Title:
{first_paper['title']}

Abstract:
{first_paper['abstract']}


Paper 2

Title:
{second_paper['title']}

Abstract:
{second_paper['abstract']}


Compare them based on:

1. Research Objective

2. Methodology

3. Key Contributions

4. Advantages

5. Limitations

6. Applications

Present the comparison in a clear table.
"""


    response = llm.invoke(comparison_prompt)

    return response.content